# Sentinel-1 Fast Ice Detection using STAC and xarray

## Data sources

Two Sentinel-1 data sources are supported and can be used independently or combined:

| | **EW (Extra Wide)** | **IW (Interferometric Wide)** |
|---|---|---|
| **Source** | STAC Geoparquet | STAC API (`pystac-client`) |
| **Collection** | `ga_s1_nrb_ew_hh_hv_1` | `ga_s1_nrb_iw_hh_1` |
| **Band loaded** | `hh_gamma0` (linear) | `hh_gamma0` (linear) |

## Pipeline overview

1. **Define region and baseline** — specify a region of interest (ROI) and temporal baseline (e.g. ~12-day).
2. **Load data** — retrieve both EW and IW Sentinel-1 data for the ROI.
3. **Pre-process** — filter and select valid image pairs meeting selection criteria.
4. **Compute NormCovar** — calculate the normalised covariance between image pairs. NormCovar is a windowed, normalised measure of how strongly two SAR images vary together at each pixel, computed by dividing local covariance by the square root of local variance. Output is an upsampled **RGB** image.
5. **SAM - Setup** — initialise the Segment Anything Model, using GPU acceleration if available.
6. **SAM - Segment** — pass the RGB image to SAM to generate image segments.
7. **SVM - Setup** - Set up a Support Vector Machine (SVM) classifier. Retrieve and train if weights don't exist.
8. **SVM - Classify image segments** - Perform binary classification of image segments as **fast ice** / not fast ice.
9. **Results - Write to file** - Save the prediction to a geotiff.

## 1. Imports

In [ ]:
import os
import fsspec
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import stac_geoparquet
import xarray as xr
from pathlib import Path
import urllib.request
from pathlib import Path
import rasterio
import rioxarray

from odc.geo import BoundingBox
from odc.stac import load, configure_s3_access
from pystac_client import Client

from normalized_covariance import normcovar
from normalized_covariance.xarray_utils import apply_lee_filter

os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## 2. Parameters

In [ ]:
# ---- Shared spatial / temporal parameters ----

# Area of interest 
### 1. Erebus Ice Tongue, McMurdo Sound (EPSG:4326)
AOI_BBOX = BoundingBox(
    left=165.5,
    bottom=-77.85,
    right=168.0,
    top=-77.45,
    crs="EPSG:4326",
)
START_TIME = "2019-01-01"
END_TIME   = "2020-01-01"

#### 2. Prydz Bay (EPSG:4326)
# AOI_BBOX = BoundingBox(
#     left=74.2,
#     bottom=-69.5,
#     right=75.0,
#     top=-69.1,
#     crs="EPSG:4326",
# ) 
# START_TIME = "2021-01-01"
# END_TIME   = "2022-01-31"

# ---- Load settings ----
OUTPUT_CRS = "EPSG:3031"
RESOLUTION = 40 
ORBIT_STATE = None # "ascending", "descending" or None

# ---- Optional Pre-processing ----

CONVERT_TO_DB = True # convert from linear to db before NormCovar process
APPLY_LEE_SPECKLE_FILTER = False # apply a speckle filter before NormCovar process
LEE_SPECKLE_FILTER_WINDOW_SIZE = 7 # size of window for speckle filter

# ---- NormCovar parameters ----

WINDOWS           = [11, 21, 33]   # three required for RGB stack
MIN_TEMP_BASELINE = 11.9           # days
MAX_TEMP_BASELINE = 12.1
NP_MIN            = -0.5
NP_MAX            =  1.0

# ---- EW parameters (STAC Geoparquet) ----

EW_PARQUET_URL = "https://data.dev.dea.ga.gov.au/experimental/baseline/pyrosar_gamma/ga_s1_nrb_ew_hh_hv_1_v2.parquet"
EW_COLLECTION  = "ga_s1_nrb_ew_hh_hv_1"
EW_BAND        = "hh_gamma0" # linear scale
#EW_BAND        = "hv_gamma0" # dual pol available for EW

# ---- IW parameters (pystac-client STAC API) ----

IW_STAC_ENDPOINT = "https://explorer.dev.dea.ga.gov.au/stac"
IW_COLLECTION    = "ga_s1_nrb_iw_hh_1"
IW_BAND          = "hh_gamma0"    # linear scale

# ---- Local Folders ---- #
DATA_FOLDER =  Path("./data")

## 3. Load EW data — STAC Geoparquet

In [ ]:
with fsspec.open(EW_PARQUET_URL, mode="rb") as f:
    gdf_ew = gpd.read_parquet(f)

print(f"Total EW items in catalogue: {len(gdf_ew)}")
gdf_ew.tail(2)

In [ ]:
aoi_shape = shapely.geometry.shape(AOI_BBOX.polygon)

filtered_ew = gdf_ew[
    (gdf_ew["collection"] == EW_COLLECTION) &
    (gdf_ew.intersects(aoi_shape)) &
    (gdf_ew["datetime"] >= START_TIME) &
    (gdf_ew["datetime"] <= END_TIME)
].copy().sort_values("datetime").reset_index(drop=True)

if ORBIT_STATE:
    print(f'Filtering for orbit state : {ORBIT_STATE}')
    filtered_ew = filtered_ew[filtered_ew["sat:orbit_state"] == ORBIT_STATE]
else:
    orbit_counts = filtered_ew["sat:orbit_state"].value_counts()
    print(f"EW orbit state breakdown: {dict(orbit_counts)}")
    print("Set ORBIT_STATE = 'ascending' or 'descending' in the parameters cell to filter.")

print(f"EW items after filtering: {len(filtered_ew)}")
print(filtered_ew[["datetime", "collection"]].to_string())

In [ ]:
stac_items_ew = stac_geoparquet.to_item_collection(filtered_ew)
print(f"EW STAC items: {len(stac_items_ew)}")
print(f"EW available assets: {list(stac_items_ew[0].assets.keys())}")

ds_ew = load(
    stac_items_ew,
    bands=[EW_BAND],
    chunks={},
    resolution=RESOLUTION,
    crs=OUTPUT_CRS,
    geopolygon=aoi_shape,
)
ds_ew

In [ ]:
ds_ew[EW_BAND].isel(time=0).compute().plot.imshow(cmap="bone", vmin=0, vmax=1, figsize=(9, 5))
plt.title(f"EW {EW_BAND} — {str(ds_ew.time.values[0])[:10]}")
plt.show()

## 4. Load IW data — pystac-client STAC API

In [ ]:
stac_client = Client.open(IW_STAC_ENDPOINT)
print(f"Searching collection: {IW_COLLECTION}")

In [ ]:
# NOTE - IW items are individual bursts so there will be more items than EW scenes
iw_search = stac_client.search(
    collections=[IW_COLLECTION],
    datetime=f"{START_TIME}/{END_TIME}",
    intersects=AOI_BBOX.boundary(),
)
stac_items_iw = iw_search.item_collection()
print(f"IW items found: {len(stac_items_iw)}")
print(f"IW assets: {list(stac_items_iw[0].assets.keys())}")

In [ ]:
# Optional: filter by orbit state (ascending / descending)
# Mixing orbit states in a pair could impact results
if ORBIT_STATE is not None:
    stac_items_iw = [
        item for item in stac_items_iw
        if item.properties.get("sat:orbit_state") == ORBIT_STATE
    ]
    print(f"IW items after filtering to '{ORBIT_STATE}': {len(stac_items_iw)}")
else:
    from collections import Counter
    orbit_counts = Counter(item.properties.get("sat:orbit_state") for item in stac_items_iw)
    print(f"IW orbit state breakdown: {dict(orbit_counts)}")
    print("Set ORBIT_STATE = 'ascending' or 'descending' in the parameters cell to filter.")

In [ ]:
# Load IW data — env ensures unsigned S3 access and retries are active at read time
ds_iw = load(
    stac_items_iw,
    bands=[IW_BAND],
    chunks={},
    resolution=RESOLUTION,
    crs=OUTPUT_CRS,
    geopolygon=aoi_shape,
    groupby="solar_day",
)
print(ds_iw)

In [ ]:
ds_iw[IW_BAND].isel(time=0).compute().plot.imshow(cmap="bone", robust=True, figsize=(10, 5))
plt.title(f"IW {IW_BAND} (linear) — {str(ds_iw.time.values[0])[:10]}")
plt.show()

## 5. PRE-Process Compare and combine EW and IW

Both datasets are loaded at the same CRS and resolution onto the same spatial grid, so they can be directly compared or merged.

**Unit alignment:** The preferred scaling for NormCovar is decibels. Convert both to dB before combining.
**Optional:** Apply a speckle filter

In [ ]:
# Convert both to dB
if APPLY_LEE_SPECKLE_FILTER:
    print(f'Warning: Applying speckle filter')
    ds_iw["hh_gamma0_filtered"] = apply_lee_filter(ds_iw.hh_gamma0, size=LEE_SPECKLE_FILTER_WINDOW_SIZE)
    ds_ew["hh_gamma0_filtered"] = apply_lee_filter(ds_ew.hh_gamma0, size=LEE_SPECKLE_FILTER_WINDOW_SIZE)
    IW_BAND = "hh_gamma0_filtered"
    EW_BAND = "hh_gamma0_filtered"

if CONVERT_TO_DB:
    print(f'Warning: Converting to dB')
    iw_db = 10 * np.log10(ds_iw[IW_BAND].where(ds_iw[IW_BAND] > 0))
    ds_iw_db = ds_iw.assign({"hh_gamma0_db": iw_db}).drop_vars(IW_BAND)
    ew_db = 10 * np.log10(ds_ew[EW_BAND].where(ds_ew[EW_BAND] > 0))
    ds_ew_db = ds_ew.assign({"hh_gamma0_db": ew_db}).drop_vars(EW_BAND)
    IW_BAND = "hh_gamma0_db"
    EW_BAND = "hh_gamma0_db"

In [ ]:
# Plot a co-located EW and IW acquisition side by side
ew_times = pd.DatetimeIndex(ds_ew_db.time.values)
iw_times = pd.DatetimeIndex(ds_iw_db.time.values)

if len(iw_times) > 0:
    closest_iw_idx = np.argmin(np.abs(iw_times - ew_times[0]))
    ew_slice = ds_ew_db[EW_BAND].isel(time=0).compute()
    iw_slice = ds_iw_db[IW_BAND].isel(time=closest_iw_idx).compute()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ew_slice.plot.imshow(ax=axes[0], cmap="bone", vmin=-20, vmax=0, add_colorbar=True)
    axes[0].set_title(f"EW {EW_BAND}\n{str(ew_times[0])[:10]}")
    iw_slice.plot.imshow(ax=axes[1], cmap="bone", vmin=-20, vmax=0, add_colorbar=True)
    axes[1].set_title(f"IW {IW_BAND}\n{str(iw_times[closest_iw_idx])[:10]}")
    plt.tight_layout()
    plt.show()
else:
    print("No IW data loaded — check IW_STAC_ENDPOINT, IW_COLLECTION, and AOI/time parameters.")

## 6. Select dataset and find valid image pairs

Choose which dataset to run NormCovar on: `"ew"`, `"iw"`, or `"both"`.

In [ ]:
# Select dataset for NormCovar processing
# Options: "ew", "iw", "both"
PROCESS_MODE = "ew"

COMBINED_BAND = "hh_gamma0_combined"  # common band name used when merging

if PROCESS_MODE == "ew":
    ds_proc   = ds_ew_db
    band_proc = EW_BAND

elif PROCESS_MODE == "iw":
    ds_proc   = ds_iw_db
    band_proc = IW_BAND

elif PROCESS_MODE == "both":
    ew_renamed = ds_ew_db.rename({EW_BAND: COMBINED_BAND}).assign_coords(source="ew")
    iw_renamed = ds_iw_db.rename({IW_BAND: COMBINED_BAND}).assign_coords(source="iw")
    ds_proc = (
        xr.concat([ew_renamed, iw_renamed], dim="time")
        .sortby("time")
    )
    band_proc = COMBINED_BAND

else:
    raise ValueError(f"Unknown PROCESS_MODE '{PROCESS_MODE}'. Choose 'ew', 'iw', or 'both'.")

print(f"Mode      : {PROCESS_MODE}")
print(f"Band      : {band_proc}")
print(f"Timesteps : {len(ds_proc.time)}")
print(ds_proc)

In [ ]:
times = pd.DatetimeIndex(ds_proc.time.values)

# Get source mode per timestep if available (only set in "both" mode)
if "source" in ds_proc.coords:
    sources = ds_proc.source.values
else:
    sources = np.full(len(times), PROCESS_MODE)

valid_pairs = []
for i in range(len(times)):
    for j in range(i + 1, len(times)):
        delta_days = abs((times[j] - times[i]).total_seconds()) / 86400
        if MIN_TEMP_BASELINE <= delta_days <= MAX_TEMP_BASELINE:
            valid_pairs.append((i, j, delta_days))

print(f"Found {len(valid_pairs)} valid pair(s) with temporal baseline {MIN_TEMP_BASELINE}-{MAX_TEMP_BASELINE} days:")
for i, j, d in valid_pairs:
    print(f"  [{sources[i].upper()}] {str(times[i])[:19]}  ↔  [{sources[j].upper()}] {str(times[j])[:19]}  ({d:.2f} days)")

## 7. Select and plot a pair

In [ ]:
if not valid_pairs:
    raise RuntimeError(
        "No valid pairs found. Try widening START_TIME / END_TIME, relaxing the temporal baseline "
        "tolerance, or switching PROCESS_MODE."
    )

test_index = 0
idx1, idx2, baseline = valid_pairs[test_index]
date1 = str(times[idx1])[:10]
date2 = str(times[idx2])[:10]
mode1 = sources[idx1].upper()
mode2 = sources[idx2].upper()

print(f"Selected pair:")
print(f"  Image 1 [{mode1}] (idx={idx1}): {date1}")
print(f"  Image 2 [{mode2}] (idx={idx2}): {date2}")
print(f"  Temporal baseline: {baseline:.2f} days")

img1 = ds_proc[band_proc].isel(time=idx1).drop_vars("time").compute()
img2 = ds_proc[band_proc].isel(time=idx2).drop_vars("time").compute()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img, date, mode, n in zip(axes, [img1, img2], [date1, date2], [mode1, mode2], [1, 2]):
    img.plot.imshow(ax=ax, cmap="bone", vmin=-20, vmax=0, add_colorbar=True)
    ax.set_title(f"Image {n} [{mode}] — {date}")
    ax.set_aspect("equal")
plt.suptitle(f"baseline: {baseline:.2f} days", y=1.02)
plt.tight_layout()
plt.show()

## 8. Process the image pair

A single call to `normcovar.fully_process_image_pair_xr` computes the Normalised Covariance between the imags. NormCovar is a windowed, normalised measure of how strongly two SAR images vary together at each pixel, computed by dividing local covariance by the square root of local variance. Output is a dictionary of data produced from the process. 

Possible output keys are `normcovar`, `rgb`, `rgb_resampled`, `landmask`, `landmask_resampled`, `rgb_landmasked`, `rgb_resampled_landmasked`. An explanation on these can be found [here](https://github.com/jlo031/normalized_product/blob/f416dac7ca7699a34c38c19c76de2696a844d896/src/normalized_product/normcovar.py#L759).

The `rgb_sampled` or `rgb_resampled_landmasked` (if a landmask is applied) should be used as input to the SAM model.

The cell below will fetch and unzip the SCAR ADD medium-res coastline shapefile

In [ ]:
from fast_ice.datasets.coastline import fetch_add_coastline_shapefile
landmask_shapefile_path = fetch_add_coastline_shapefile(output_dir=DATA_FOLDER)

In [ ]:
results = normcovar.fully_process_image_pair_xr(
    img1,
    img2,
    windows=WINDOWS,
    NP_min=NP_MIN,
    NP_max=NP_MAX,
    resample_rgb=True,
    resample_method="bilinear",
    zoom_x=10,
    zoom_y=10,
    landmask_shapefile_path=landmask_shapefile_path,
    erode_landmask=None,
    apply_landmask_to_rgb=True
)

print(results.keys())

## 9. Visualise NormCovar results

In [ ]:
fig, axes = plt.subplots(1, len(WINDOWS), figsize=(6 * len(WINDOWS), 5))

for ax, window in zip(axes, WINDOWS):
    results["normcovar"][window].plot.imshow(
        ax=ax, cmap="RdYlBu_r", vmin=NP_MIN, vmax=NP_MAX, add_colorbar=True,
    )
    ax.set_title(f"normcovar  w={window}\n[{mode1} / {mode2}]  {date1} ↔ {date2}")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 10. False-colour RGB composite

In [ ]:
if "rgb" not in results:
    print("RGB composite requires exactly 3 window sizes — skipping.")
else:
    ref = results["normcovar"][WINDOWS[0]]
    x_min, x_max = float(ref.x.min()), float(ref.x.max())
    y_min, y_max = float(ref.y.min()), float(ref.y.max())

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(results["rgb"], extent=[x_min, x_max, y_min, y_max], origin="upper")
    ax.set_title(
        f"normcovar RGB  [{mode1} / {mode2}]  (R=w{WINDOWS[0]}, G=w{WINDOWS[1]}, B=w{WINDOWS[2]})\n"
        f"{date1} ↔ {date2}  |  {OUTPUT_CRS}"
    )
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    plt.tight_layout()
    plt.show()

    # Print RGB and Landmasked / Resampled
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ref = results["normcovar"][WINDOWS[0]]
    x_min, x_max = float(ref.x.min()), float(ref.x.max())
    y_min, y_max = float(ref.y.min()), float(ref.y.max())
    extent = [x_min, x_max, y_min, y_max]

    rgb_resampled = results["rgb_resampled"]
    landmask_resampled = results["landmask_resampled"]
    rgb_resampled_landmasked = results["rgb_resampled_landmasked"]

    # Panel 1: resampled RGB
    axes[0].imshow(rgb_resampled, extent=extent, origin="upper")
    axes[0].set_title("normcovar RGB (resampled)")

    # Panel 2: resampled landmask
    axes[1].imshow(landmask_resampled, extent=extent, origin="upper", cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Landmask (resampled)")

    # Panel 3: RGB with landmask applied
    axes[2].imshow(rgb_resampled_landmasked, extent=extent, origin="upper", cmap="gray")
    axes[2].set_title("normcovar RGB (resampled and Landmasked)")

    for ax in axes:
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")
        ax.set_aspect("equal")

    fig.suptitle(f"[{mode1} / {mode2}]  {date1} ↔ {date2}  |  {OUTPUT_CRS}", y=1.03)
    plt.tight_layout()
    plt.show()

## 11. (Optional) Process all valid pairs

In [ ]:
# all_results = []

# for idx1, idx2, baseline in valid_pairs:
#     date1 = str(times[idx1])[:10]
#     date2 = str(times[idx2])[:10]
#     mode1 = sources[idx1].upper()
#     mode2 = sources[idx2].upper()
#     print(f"Processing [{mode1}] {date1} ↔ [{mode2}] {date2} ({baseline:.2f} days)")

#     img1 = ds_proc[band_proc].isel(time=idx1).drop_vars("time").compute()
#     img2 = ds_proc[band_proc].isel(time=idx2).drop_vars("time").compute()

#     pair_results = normcovar.fully_process_image_pair_xr(
#         img1, img2,
#         windows=WINDOWS,
#         NP_min=NP_MIN,
#         NP_max=NP_MAX,
#     )

#     all_results.append({
#         "date1": date1, "date2": date2,
#         "baseline_days": baseline,
#         "mode": f"{mode1}/{mode2}",
#         "normcovar": pair_results,
#     })

# print(f"Processed {len(all_results)} pair(s).")

## 12. Run SAM Segmentation

The SAM setup/inference logic below (imports, weight download, model + mask generator setup, mask generation, and non-overlapping segment ID map construction) has been moved into `fast_ice/sam/` as a reusable `SAMSegmenter` class, with plotting helpers in `fast_ice.sam.predict`. This cell is equivalent to the manual walkthrough in the following cells.

In [ ]:
from fast_ice.sam.model import SAMSegmenter
from fast_ice.sam.predict import annotate_masks, plot_segmentation, plot_segment_id_map
from fast_ice.sam.constants import FAST_MASK_GENERATOR_SETTINGS

sam_input = results["rgb_resampled_landmasked"].values.astype(np.uint8)

# Downloads weights (if needed) and loads the model + mask generator on first use.
# or pass mask_generator_kwargs={...} to override individual settings.
segmenter = SAMSegmenter(model_type="vit_h", mask_generator_settings=FAST_MASK_GENERATOR_SETTINGS)
segment_id_map, sam_result = segmenter.predict(sam_input)

annotated = annotate_masks(sam_input, sam_result)
plot_segmentation(sam_input, annotated, title=f"dates: {date1}_{date2}")
plot_segment_id_map(segment_id_map, title=f"Segment ID map — smallest area wins | dates: {date1}_{date2}")

## 13. SVM - Download and Train the SVM if Weights Don't Exist

In [ ]:
from fast_ice.svm.constants import FAST_ICE_CLASS_VALUES
from fast_ice.datasets.svm_training_data import fetch_svm_trainingdata
from fast_ice.svm.train import train

SVM_WEIGHTS = '../fast_ice/svm/weights/2026_08_02_svm_weights.joblib'
SVM_TRAINING_DATA_FOLDER = 'data'

if not Path(SVM_WEIGHTS).exists():
    svm_training_data_dir = fetch_svm_trainingdata(SVM_TRAINING_DATA_FOLDER)
    print(svm_training_data_dir)
    train(svm_training_data_dir, SVM_WEIGHTS)
# train('data/SVM_trainingdata/SVM_trainingdata', SVM_WEIGHTS)

## 14. Classify SAM Clusters with SVM 

In [ ]:
from fast_ice.svm.predict import predict_on_image_and_segments, plot_predictions
pred_map, segment_class = predict_on_image_and_segments(sam_input, segment_id_map, model=SVM_WEIGHTS)
plot_predictions(sam_input, pred_map)

## 15. Convert output to a binary mask and save as Geotiff

In [ ]:
from fast_ice.utils import assign_geodata_from_xr
pred_map_xr = assign_geodata_from_xr(pred_map, results["rgb_resampled_landmasked"])
binary_fast_ice_xr = xr.where(pred_map_xr.isin(FAST_ICE_CLASS_VALUES), 1, 0)
binary_fast_ice_xr = binary_fast_ice_xr.rio.write_crs(pred_map_xr.rio.crs)
binary_fast_ice_xr = binary_fast_ice_xr.rio.write_transform(pred_map_xr.rio.transform())
binary_fast_ice_xr = binary_fast_ice_xr.astype(np.uint8)

In [ ]:
tif_name = f"{date1}_{date2}_{AOI_BBOX.left}_{AOI_BBOX.bottom}_{AOI_BBOX.right}_{AOI_BBOX.top}_fast_ice.tif"
binary_fast_ice_xr.rio.to_raster(tif_name, compress="lzw")